### 🧭 **Topic Modeling**

Este guia descreve um fluxo prático e reproduzível para modelagem de tópicos em coleções de texto.  
A ideia é evitar treinar modelos pesados desde o início, ajustando primeiro os hiperparâmetros em uma amostra menor e depois escalando para o conjunto completo.

---

#### 📑 **Fluxo Geral**

1. **Separar uma amostra** para ajuste de parâmetros.  
2. **Treinar o modelo** em uma parte maior do conjunto usando os parâmetros ajustados.  
3. **Classificar o restante do corpus** com os tópicos aprendidos.

---


In [2]:
'''Carrega o CSV, removendo linhas mal formatadas'''
import pandas as pd

file_path = "../data/df_clean_text.csv"
data_frame = pd.read_csv(file_path)
print(f"Total documents loaded: {len(data_frame)}")

Total documents loaded: 6506


In [3]:
'''Remoção de Stopwords'''
from PreProcessing.pre_processing import PreProcessing
from nltk.corpus import stopwords

pp = PreProcessing(language="pt")

custom_stopwords = [line.strip() for line in open('stopwords.txt', 'r', encoding='utf-8')]
pt_stopwords = set(stopwords.words('portuguese'))
pp.append_stopwords_list(list(pt_stopwords - set(pp.stopwords)) + custom_stopwords)

data_frame["clean_text"] = data_frame["clean_text"].apply(pp.remove_stopwords)


In [4]:
'''Remoção de linhas nulas e duplicatas'''
nan_count = data_frame['clean_text'].isna().sum()
df = data_frame[data_frame['clean_text'].notna()].reset_index(drop=True)
print(f"{nan_count} rows with NaN in 'clean_text' were removed.")

duplicate_count = df.duplicated(subset=['clean_text']).sum()
df.drop_duplicates(subset=['clean_text'], inplace=True)
print(f"{duplicate_count} duplicate rows based on 'clean_text' were removed.")

print(f"Total documents after cleaning: {len(df)}")

0 rows with NaN in 'clean_text' were removed.
76 duplicate rows based on 'clean_text' were removed.
Total documents after cleaning: 6430


In [5]:
docs = df['clean_text'].tolist()
print(f"{len(docs)} documents for topic modeling.")

6430 documents for topic modeling.


In [6]:
# Defining parameters for topic modeling

"""
UMAP PARAMETERS

n_neighbors: Controla o equilíbrio entre a preservação da estrutura global e local dos dados.
n_components: A dimensionalidade do espaço onde os clusters são formados. 
              Um espaço de menor dimensão pode forçar os pontos a se agruparem de forma mais densa.

HDBSCAN PARAMETERS
min_cluster_size: O tamanho mínimo de um cluster. Clusters menores que esse valor serão considerados ruído.
min_samples: Influencia a sensibilidade do algoritmo à densidade dos pontos.
             Valores maiores consideram apenas áreas muito densas como clusters.
cluster_eps: Define a distância máxima entre pontos para que sejam considerados parte do mesmo cluster.
             
EMBEDDING MODELS
Modelos de linguagem pré-treinados usados para gerar embeddings dos textos.
"""

umap_neighbors = [10, 20]
umap_components = [5, 10]
umap_min_dist = [0.0, 0.1, 0.5]

hdbscan_min_cluster = [50, 100, 150]
hdbscan_min_samples = [5, 10, 100]
hdbscan_cluter_eps = [0.1]

embedding_models = [
    #"all-MiniLM-L6-v2", 
    #"all-distilroberta-v1", 
    "paraphrase-MiniLM-L6-v2", 
    "neuralmind/bert-base-portuguese-cased"
]

default_stopwords = []

results = []

In [7]:
import os

local_cache_dir = os.path.join(os.getcwd(), '.hf_cache')
os.makedirs(local_cache_dir, exist_ok=True)

os.environ['HF_HOME'] = local_cache_dir
os.environ['HUGGINGFACE_HUB_CACHE'] = local_cache_dir
os.environ['TRANSFORMERS_CACHE'] = local_cache_dir

from sentence_transformers import SentenceTransformer

print(f"Cache travado em: {local_cache_dir}")
print("Iniciando o download e carregamento dos modelos na GPU...")

#embedding_all_Mini = SentenceTransformer("all-MiniLM-L6-v2", device='cuda')
#print("1/4: all-MiniLM carregado!")

#embedding_roberta = SentenceTransformer("all-distilroberta-v1", device='cuda')
#print("2/4: distilroberta carregado!")

embedding_paraphrase = SentenceTransformer("paraphrase-MiniLM-L6-v2", device='cuda')
print("3/4: paraphrase-MiniLM carregado!")

embedding_bertimbau = SentenceTransformer("neuralmind/bert-base-portuguese-cased", device='cuda')
print("4/4: BERTimbau carregado!")

print("\nTodos os modelos estão prontos e salvos na sua pasta local!")

Cache travado em: /home/voamorim/Documents/UFSJ/6o_periodo/DataMining/LeiFelca/notebooks/.hf_cache
Iniciando o download e carregamento dos modelos na GPU...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10842.35it/s]


3/4: paraphrase-MiniLM carregado!


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 52947.63it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4/4: BERTimbau carregado!

Todos os modelos estão prontos e salvos na sua pasta local!


In [8]:
import nltk

try:
    # Tenta baixar o recurso 'punkt_tab'. Se já estiver baixado, ignora.
    nltk.download('punkt_tab')
except LookupError:
    # Se 'punkt_tab' não funcionar (depende da versão do NLTK), 
    # use o pacote 'punkt' mais genérico.
    # O seu traceback pede 'punkt_tab', então essa é a melhor aposta.
    # No entanto, se o NLTK for muito antigo, pode ser só 'punkt'.
    nltk.download('punkt')

# Se você estiver usando stopwords personalizadas, pode precisar disso também:
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/voamorim/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/voamorim/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
from itertools import product
from sklearn.feature_extraction.text import TfidfVectorizer
from meu_bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from nltk.tokenize import word_tokenize
import numpy as np
from sklearn.metrics import silhouette_score as compute_silhouette_score
from tqdm.auto import tqdm

# Tokenizar os documentos (necessário para Gensim)
tokenized_docs = [word_tokenize(doc.lower()) for doc in docs]

# Cria o Dicionário e Corpus BoW para Gensim 
dictionary = Dictionary(tokenized_docs)
corpus_gensim = [dictionary.doc2bow(text) for text in tokenized_docs]

# Pré-calcula os embeddings
embeddings_dict = {
    #"all-MiniLM-L6-v2": embedding_all_Mini.encode(docs, show_progress_bar=True),
    "paraphrase-MiniLM-L6-v2": embedding_paraphrase.encode(docs, show_progress_bar=True), 
    "neuralmind/bert-base-portuguese-cased": embedding_bertimbau.encode(docs, show_progress_bar=True),
    #"all-distilroberta-v1": embedding_roberta.encode(docs, show_progress_bar=True)
}

results = []

for i, (n_neighbors, n_components, min_dist, min_cluster_size, min_samples, cluster_eps, embedding_model_name) in tqdm(
    enumerate(product(
        umap_neighbors,
        umap_components,
        umap_min_dist,
        hdbscan_min_cluster,
        hdbscan_min_samples,
        hdbscan_cluter_eps,
        embedding_models
    ), start=1),
    total=(len(umap_neighbors) * len(umap_components) * len(umap_min_dist) *
           len(hdbscan_min_cluster) * len(hdbscan_min_samples) * 
           len(hdbscan_cluter_eps) * len(embedding_models)),
    desc="Treinando modelos"
):    
    umap_params = {
        "n_neighbors": n_neighbors, 
        "n_components": n_components, 
        "min_dist": min_dist,
        "metric":'cosine',
        "random_state":42
    }

    hdbscan_params = {
        "min_cluster_size": min_cluster_size, 
        "min_samples": min_samples,
        "prediction_data":True,
        "cluster_selection_epsilon": cluster_eps
    }

    bertopic_params = {
        'language': 'portuguese',
        #'verbose': True,
        'top_n_words': 20
    }

    # Recupera o embedding pré-calculado
    current_embeddings = embeddings_dict[embedding_model_name]

    vectorizer = TfidfVectorizer(stop_words=default_stopwords,ngram_range=(1, 2))
    umap_model = UMAP(**umap_params)
    hdbscan_model = HDBSCAN(**hdbscan_params)

    model = BERTopic(
        **bertopic_params,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer,
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True)
    )

    topics, probs = model.fit_transform(docs, embeddings=current_embeddings)

    topic_words = model.get_topics() 
    # Extrai apenas as palavras para a Gensim
    topics_for_gensim = [[word for word, score in topic_words[topic_id]] 
                        for topic_id in topic_words if topic_id != -1]

    # 4. Calcular Coerência
    try:
        coherence_model = CoherenceModel(
            topics=topics_for_gensim, 
            texts=tokenized_docs, 
            corpus=corpus_gensim,
            dictionary=dictionary, 
            coherence='c_v'  # Métrica C_V é a mais robusta
        )
        coherence_score = coherence_model.get_coherence()
    except Exception:
        coherence_score = 0.0

    # --- Fluxo de cálculo da Diversidade (Proportion of Unique Words) ---
    all_topic_words = []

    # Coleta todas as top N palavras, excluindo o tópico -1 (outliers)
    for topic_id in topic_words:
        if topic_id != -1:
            # Pega apenas as palavras (o primeiro elemento de cada tupla)
            words = [word for word, score in topic_words[topic_id]]
            all_topic_words.extend(words)
            
    total_words = len(all_topic_words)
    unique_words = len(set(all_topic_words))

    # Evita divisão por zero se o modelo não encontrar nenhum tópico (improvável no seu caso)
    if total_words > 0:
        diversity_score = unique_words / total_words
    else:
        diversity_score = 0

    try:
        # Tenta obter os embeddings reduzidos (o dado de clusterização)
        reduced_embeddings = model.umap_model.embedding_
    except:
        reduced_embeddings = model.umap_model.transform(current_embeddings)

    # O topics (rótulos de cluster) é o resultado do model.fit_transform(sample_docs)
    # Filtra os outliers (-1) do HDBSCAN
    # O score da Silhueta geralmente é calculado *apenas* sobre os pontos que foram clusterizados
    # O sk-learn tem o parâmetro 'metric' que você já usou no UMAP ('cosine').

    # 1. Filtra os dados: remove o tópico -1 (outliers do HDBSCAN)
    mask = np.array(topics) != -1
    filtered_embeddings = reduced_embeddings[mask]
    filtered_topics = np.array(topics)[mask]

    # 2. Calcula o Silhouette Score (usa a métrica de distância 'cosine' que você usou no UMAP)
    if len(np.unique(filtered_topics)) > 1: # Precisa de pelo menos 2 clusters
        silhouette_score = compute_silhouette_score(
            filtered_embeddings, 
            filtered_topics, 
            metric='cosine' # Usa a métrica consistente com o UMAP
        )
    else:
        silhouette_score = 0.0 # Não é possível calcular Silhueta com 0 ou 1 cluster

    # 2. REGISTRO DOS RESULTADOS
    results.append({
        'n_neighbors': n_neighbors,
        'n_components': n_components,
        'min_dist': min_dist,
        'min_cluster_size': min_cluster_size,
        'min_samples': min_samples,
        'cluster_eps': cluster_eps,
        'embedding_model': embedding_model_name,
        'coherence': coherence_score,
        'diversity': diversity_score,
        'silhouette': silhouette_score
    })


KeyboardInterrupt: 

### Usando CUDA

In [ ]:
from itertools import product
from sklearn.feature_extraction.text import TfidfVectorizer
from meu_bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN
from cuml.metrics.cluster import silhouette_score as cuml_silhouette_score
import cupy as cp

from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from nltk.tokenize import word_tokenize
import numpy as np
from tqdm.auto import tqdm

# Tokenizar os documentos (necessário para Gensim)
tokenized_docs = [word_tokenize(doc.lower()) for doc in docs]

# Cria o Dicionário e Corpus BoW para Gensim 
dictionary = Dictionary(tokenized_docs)
corpus_gensim = [dictionary.doc2bow(text) for text in tokenized_docs]

embeddings_dict = {
    #"all-MiniLM-L6-v2": embedding_all_Mini.encode(docs, show_progress_bar=True),
    "paraphrase-MiniLM-L6-v2": embedding_paraphrase.encode(docs, show_progress_bar=True), 
    "neuralmind/bert-base-portuguese-cased": embedding_bertimbau.encode(docs, show_progress_bar=True),
    #"all-distilroberta-v1": embedding_roberta.encode(docs, show_progress_bar=True)
}

results = []

for i, (n_neighbors, n_components, min_dist, min_cluster_size, min_samples, cluster_eps, embedding_model_name) in tqdm(
    enumerate(product(
        umap_neighbors,
        umap_components,
        umap_min_dist,
        hdbscan_min_cluster,
        hdbscan_min_samples,
        hdbscan_cluter_eps,
        embedding_models
    ), start=1),
    total=(len(umap_neighbors) * len(umap_components) * len(umap_min_dist) *
           len(hdbscan_min_cluster) * len(hdbscan_min_samples) * len(hdbscan_cluter_eps) * len(embedding_models)),
    desc="Treinando modelos"
):    
    umap_params = {
        "n_neighbors": n_neighbors, 
        "n_components": n_components, 
        "min_dist": min_dist,
        "metric": 'cosine',
        "random_state": 42
    }

    hdbscan_params = {
        "min_cluster_size": min_cluster_size, 
        "min_samples": min_samples,
        "prediction_data": True,
        "cluster_selection_epsilon": cluster_eps
    }

    bertopic_params = {
        'language': 'portuguese',
        'top_n_words': 20
    }

    # Recupera o embedding pré-calculado
    current_embeddings = embeddings_dict[embedding_model_name]

    vectorizer = TfidfVectorizer(stop_words=default_stopwords, ngram_range=(1, 2))
    
    umap_model = UMAP(**umap_params)
    hdbscan_model = HDBSCAN(**hdbscan_params)

    model = BERTopic(
        **bertopic_params,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer,
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True)
    )

    topics, probs = model.fit_transform(docs, embeddings=current_embeddings)

    if hasattr(topics, "to_numpy"):
        topics_cpu = topics.to_numpy().tolist()
    elif hasattr(topics, "get"):
        topics_cpu = topics.get().tolist()
    else:
        topics_cpu = [int(t) for t in topics]

    topic_words = model.get_topics() 
    topics_for_gensim = [[word for word, score in topic_words[topic_id]] 
                        for topic_id in topic_words if topic_id != -1]

    # Calcular Coerência 
    try:
        coherence_model = CoherenceModel(
            topics=topics_for_gensim, 
            texts=tokenized_docs, 
            corpus=corpus_gensim,
            dictionary=dictionary, 
            coherence='c_v'
        )
        coherence_score = coherence_model.get_coherence()
    except Exception:
        coherence_score = 0.0

    all_topic_words = []
    for topic_id in topic_words:
        if topic_id != -1:
            words = [word for word, score in topic_words[topic_id]]
            all_topic_words.extend(words)
            
    total_words = len(all_topic_words)
    unique_words = len(set(all_topic_words))
    diversity_score = unique_words / total_words if total_words > 0 else 0.0

    try:
        reduced_embeddings = model.umap_model.embedding_
    except:
        reduced_embeddings = model.umap_model.transform(current_embeddings)

    reduced_embeddings = cp.asarray(reduced_embeddings)
    topics_gpu = cp.array(topics_cpu)

    # 1. Filtra os dados : remove os outliers (-1)
    mask = topics_gpu != -1
    filtered_embeddings = reduced_embeddings[mask]
    filtered_topics = topics_gpu[mask]

    # 2. Calcula o Silhouette Score 
    if len(cp.unique(filtered_topics)) > 1:
        silhouette_score = cuml_silhouette_score(
            filtered_embeddings, 
            filtered_topics, 
            metric='cosine'
        )
        # Extrai o valor numérico puro do objeto 
        if hasattr(silhouette_score, "get"):
            silhouette_score = float(silhouette_score.get())
        else:
            silhouette_score = float(silhouette_score)
    else:
        silhouette_score = 0.0

    # REGISTRO DOS RESULTADOS
    results.append({
        'n_neighbors': n_neighbors,
        'n_components': n_components,
        'min_dist': min_dist,
        'min_cluster_size': min_cluster_size,
        'min_samples': min_samples,
        'cluster_eps': cluster_eps,
        'embedding_model': embedding_model_name,
        'coherence': coherence_score,
        'diversity': diversity_score,
        'silhouette': silhouette_score
    })

Treinando modelos: 100%|██████████| 432/432 [2:12:44<00:00, 18.44s/it]  


In [12]:
df_results = pd.DataFrame(results)
display(df_results)

,n_neighbors,n_components,min_dist,min_cluster_size,min_samples,cluster_eps,embedding_model,coherence,diversity,silhouette
0,10,5,0.0,50,5,0.1,all-MiniLM-L6-v2,0.539994,0.830000,0.305439
1,10,5,0.0,50,5,0.1,all-distilroberta-v1,0.481683,0.867647,-0.036027
2,10,5,0.0,50,5,0.1,paraphrase-MiniLM-L6-v2,0.454341,0.730952,0.593665
3,10,5,0.0,50,5,0.1,neuralmind/bert-base-portuguese-cased,0.601052,0.720000,0.427699
4,10,5,0.0,50,10,0.1,all-MiniLM-L6-v2,0.516302,0.916667,0.632209
...,...,...,...,...,...,...,...,...,...,...
427,20,10,0.5,150,10,0.1,neuralmind/bert-base-portuguese-cased,0.529857,0.883333,0.525170
428,20,10,0.5,150,100,0.1,all-MiniLM-L6-v2,0.482262,0.900000,0.517575
429,20,10,0.5,150,100,0.1,all-distilroberta-v1,0.482515,0.766667,0.481433
430,20,10,0.5,150,100,0.1,paraphrase-MiniLM-L6-v2,0.478247,0.850000,0.619060


In [13]:
df_results['mean_score'] = df_results[['coherence', 'diversity', 'silhouette']].mean(axis=1)
display(df_results)

,n_neighbors,n_components,min_dist,min_cluster_size,min_samples,cluster_eps,embedding_model,coherence,diversity,silhouette,mean_score
0,10,5,0.0,50,5,0.1,all-MiniLM-L6-v2,0.539994,0.830000,0.305439,0.558478
1,10,5,0.0,50,5,0.1,all-distilroberta-v1,0.481683,0.867647,-0.036027,0.437768
2,10,5,0.0,50,5,0.1,paraphrase-MiniLM-L6-v2,0.454341,0.730952,0.593665,0.592986
3,10,5,0.0,50,5,0.1,neuralmind/bert-base-portuguese-cased,0.601052,0.720000,0.427699,0.582917
4,10,5,0.0,50,10,0.1,all-MiniLM-L6-v2,0.516302,0.916667,0.632209,0.688393
...,...,...,...,...,...,...,...,...,...,...,...
427,20,10,0.5,150,10,0.1,neuralmind/bert-base-portuguese-cased,0.529857,0.883333,0.525170,0.646120
428,20,10,0.5,150,100,0.1,all-MiniLM-L6-v2,0.482262,0.900000,0.517575,0.633279
429,20,10,0.5,150,100,0.1,all-distilroberta-v1,0.482515,0.766667,0.481433,0.576871
430,20,10,0.5,150,100,0.1,paraphrase-MiniLM-L6-v2,0.478247,0.850000,0.619060,0.649102


In [14]:
df_results.to_csv("../data/bertopic_parameter_tuning_results_sample_3.csv", index=False)

### Clusterizacao usando K-means

In [ ]:
from itertools import product
from sklearn.feature_extraction.text import TfidfVectorizer
from meu_bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

from cuml.manifold import UMAP
from cuml.cluster import KMeans
from cuml.metrics.cluster import silhouette_score as cuml_silhouette_score
import cupy as cp

from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from nltk.tokenize import word_tokenize
import numpy as np
from tqdm.auto import tqdm

tokenized_docs = [word_tokenize(doc.lower()) for doc in docs]
dictionary = Dictionary(tokenized_docs)
corpus_gensim = [dictionary.doc2bow(text) for text in tokenized_docs]

embeddings_dict = {
    "paraphrase-MiniLM-L6-v2": embedding_paraphrase.encode(docs, show_progress_bar=True), 
    "neuralmind/bert-base-portuguese-cased": embedding_bertimbau.encode(docs, show_progress_bar=True)
}

kmeans_clusters = [2, 3, 4, 5, 10, 20, 30, 50] 

results_kmeans = [] # Lista separada para os novos resultados

for i, (n_neighbors, n_components, min_dist, n_clusters, embedding_model_name) in tqdm(
    enumerate(product(
        umap_neighbors,
        umap_components,
        umap_min_dist,
        kmeans_clusters, 
        embedding_models
    ), start=1),
    total=(len(umap_neighbors) * len(umap_components) * len(umap_min_dist) *
           len(kmeans_clusters) * len(embedding_models)),
    desc="Treinando modelos KMeans"
):    
    umap_params = {
        "n_neighbors": n_neighbors, 
        "n_components": n_components, 
        "min_dist": min_dist,
        "metric": 'cosine',
        "random_state": 42
    }

    kmeans_params = {
        "n_clusters": n_clusters,
        "random_state": 42
    }

    bertopic_params = {
        'language': 'portuguese',
        'top_n_words': 20
    }

    current_embeddings = embeddings_dict[embedding_model_name]
    vectorizer = TfidfVectorizer(stop_words=default_stopwords, ngram_range=(1, 2))
    
    umap_model = UMAP(**umap_params)
    kmeans_model = KMeans(**kmeans_params)

    model = BERTopic(
        **bertopic_params,
        umap_model=umap_model,
        hdbscan_model=kmeans_model, 
        vectorizer_model=vectorizer,
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True)
    )

    topics, probs = model.fit_transform(docs, embeddings=current_embeddings)

    if hasattr(topics, "to_numpy"):
        topics_cpu = topics.to_numpy().tolist()
    elif hasattr(topics, "get"):
        topics_cpu = topics.get().tolist()
    else:
        topics_cpu = [int(t) for t in topics]

    topic_words = model.get_topics() 
    topics_for_gensim = [[word for word, score in topic_words[topic_id]] 
                        for topic_id in topic_words]

    try:
        coherence_model = CoherenceModel(
            topics=topics_for_gensim, 
            texts=tokenized_docs, 
            corpus=corpus_gensim,
            dictionary=dictionary, 
            coherence='c_v'
        )
        coherence_score = coherence_model.get_coherence()
    except Exception:
        coherence_score = 0.0

    # --- Fluxo de cálculo da Diversidade ---
    all_topic_words = []
    for topic_id in topic_words:
        words = [word for word, score in topic_words[topic_id]]
        all_topic_words.extend(words)
            
    total_words = len(all_topic_words)
    unique_words = len(set(all_topic_words))
    diversity_score = unique_words / total_words if total_words > 0 else 0.0

    try:
        reduced_embeddings = model.umap_model.embedding_
    except:
        reduced_embeddings = model.umap_model.transform(current_embeddings)

    reduced_embeddings = cp.asarray(reduced_embeddings)
    topics_gpu = cp.array(topics_cpu)

    if len(cp.unique(topics_gpu)) > 1:
        silhouette_score = cuml_silhouette_score(
            reduced_embeddings, 
            topics_gpu, 
            metric='cosine'
        )
        if hasattr(silhouette_score, "get"):
            silhouette_score = float(silhouette_score.get())
        else:
            silhouette_score = float(silhouette_score)
    else:
        silhouette_score = 0.0

    # REGISTRO DOS RESULTADOS 
    results_kmeans.append({
        'n_neighbors': n_neighbors,
        'n_components': n_components,
        'min_dist': min_dist,
        'n_clusters': n_clusters, 
        'embedding_model': embedding_model_name,
        'coherence': coherence_score,
        'diversity': diversity_score,
        'silhouette': silhouette_score
    })

Treinando modelos KMeans: 100%|██████████| 192/192 [57:39<00:00, 18.02s/it] 


In [11]:
df_results_kmeans = pd.DataFrame(results_kmeans)
display(df_results_kmeans)

,n_neighbors,n_components,min_dist,n_clusters,embedding_model,coherence,diversity,silhouette
0,10,5,0.0,2,paraphrase-MiniLM-L6-v2,0.428547,0.800000,0.087046
1,10,5,0.0,2,neuralmind/bert-base-portuguese-cased,0.445422,0.875000,0.661777
2,10,5,0.0,3,paraphrase-MiniLM-L6-v2,0.447102,0.900000,0.085774
3,10,5,0.0,3,neuralmind/bert-base-portuguese-cased,0.428361,0.816667,0.472452
4,10,5,0.0,4,paraphrase-MiniLM-L6-v2,0.428067,0.687500,0.269211
...,...,...,...,...,...,...,...,...
187,20,10,0.5,20,neuralmind/bert-base-portuguese-cased,0.549001,0.670000,0.218300
188,20,10,0.5,30,paraphrase-MiniLM-L6-v2,0.460833,0.691667,0.228615
189,20,10,0.5,30,neuralmind/bert-base-portuguese-cased,0.505575,0.748333,0.201579
190,20,10,0.5,50,paraphrase-MiniLM-L6-v2,0.452815,0.799000,0.243123


In [13]:
df_results_kmeans['mean_score'] = df_results_kmeans[['coherence', 'diversity', 'silhouette']].mean(axis=1)
display(df_results_kmeans)

,n_neighbors,n_components,min_dist,n_clusters,embedding_model,coherence,diversity,silhouette,mean_score
0,10,5,0.0,2,paraphrase-MiniLM-L6-v2,0.428547,0.800000,0.087046,0.438531
1,10,5,0.0,2,neuralmind/bert-base-portuguese-cased,0.445422,0.875000,0.661777,0.660733
2,10,5,0.0,3,paraphrase-MiniLM-L6-v2,0.447102,0.900000,0.085774,0.477626
3,10,5,0.0,3,neuralmind/bert-base-portuguese-cased,0.428361,0.816667,0.472452,0.572493
4,10,5,0.0,4,paraphrase-MiniLM-L6-v2,0.428067,0.687500,0.269211,0.461593
...,...,...,...,...,...,...,...,...,...
187,20,10,0.5,20,neuralmind/bert-base-portuguese-cased,0.549001,0.670000,0.218300,0.479100
188,20,10,0.5,30,paraphrase-MiniLM-L6-v2,0.460833,0.691667,0.228615,0.460371
189,20,10,0.5,30,neuralmind/bert-base-portuguese-cased,0.505575,0.748333,0.201579,0.485162
190,20,10,0.5,50,paraphrase-MiniLM-L6-v2,0.452815,0.799000,0.243123,0.498313


In [14]:
df_results_kmeans.to_csv("../data/bertopic_parameter_tuning_results_sample_kmeans.csv", index=False)